# Fine-tune Government AI Qwen3-4B LoRA

1. Setup Kaggle + copy dataset/config
2. Convert ChatML raw dataset sang SFT text
3. Load Qwen3-4B 4-bit + LoRA
4. Train full adapter
5. Save adapter
6. Evaluate validation/test bằng parser metrics
7. Smoke test thủ công
8. Zip artifacts để dùng ở notebook inference/service


In [ ]:
import os
import sys
import subprocess

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "transformers>=4.51.0",
    "accelerate",
    "datasets",
    "tokenizers",
    "huggingface_hub",
    "sentencepiece",
    "protobuf>=5.29.1,<6",
    "peft",
    "trl",
    "bitsandbytes",
    "scikit-learn",
    "jsonschema",
])

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "wandb"],
    check=False,
)

print("Dependencies installed. W&B disabled/uninstalled.")

In [ ]:
import os
import gc
import json
import math
import time
import shutil
import zipfile
from pathlib import Path
from collections import Counter
from statistics import mean

import torch
import jsonschema
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from sklearn.metrics import accuracy_score

BASE_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

PROJECT_DIR = Path("/kaggle/working/government-parser-finetune")
DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
SFT_DATA_DIR = DATA_DIR / "sft"
CONFIG_DIR = PROJECT_DIR / "configs"
OUTPUT_DIR = PROJECT_DIR / "outputs"
REPORT_DIR = OUTPUT_DIR / "reports"

HF_CACHE_DIR = PROJECT_DIR / "hf_cache"

for p in [RAW_DATA_DIR, SFT_DATA_DIR, CONFIG_DIR, OUTPUT_DIR, REPORT_DIR, HF_CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_DIR / "transformers")
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE_DIR / "datasets")

TRAIN_MAX_LENGTH = 256

EVAL_MAX_SAMPLES = 500
EVAL_BATCH_SIZE = 8
GEN_MAX_NEW_TOKENS = 192

# Train config
NUM_TRAIN_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01

SYSTEM_PROMPT = (
    "You are a semantic parser for a Government Economic Indicator AI Agent. "
    "Output only valid JSON. Do not answer the question."
)

RAW_TRAIN_PATH = RAW_DATA_DIR / "parser_train.v1.jsonl"
RAW_VALIDATION_PATH = RAW_DATA_DIR / "parser_validation.v1.jsonl"
RAW_TEST_PATH = RAW_DATA_DIR / "parser_test.v1.jsonl"

SFT_TRAIN_PATH = SFT_DATA_DIR / "parser_train.sft.v1.jsonl"
SFT_VALIDATION_PATH = SFT_DATA_DIR / "parser_validation.sft.v1.jsonl"
SFT_TEST_PATH = SFT_DATA_DIR / "parser_test.sft.v1.jsonl"

SCHEMA_PATH = CONFIG_DIR / "parsed_query_schema.v1.json"
INTENTS_PATH = CONFIG_DIR / "parser_intents.v1.json"
ENUMS_PATH = CONFIG_DIR / "parser_enums.v1.json"
QUESTION_FAMILIES_PATH = CONFIG_DIR / "question_families.v1.json"
COUNTRY_CATALOG_PATH = CONFIG_DIR / "country_catalog.v1.json"
INDICATOR_CATALOG_PATH = CONFIG_DIR / "indicator_catalog.v1.json"
ANALYTICS_METADATA_PATH = CONFIG_DIR / "analytics_metadata.v1.json"

FULL_TRAIN_OUTPUT_DIR = OUTPUT_DIR / "parser_qwen3_4b_lora_full_train"
FINAL_ADAPTER_DIR = OUTPUT_DIR / "parser_qwen3_4b_lora_final_adapter"
EXPORT_DIR = OUTPUT_DIR / "export"
ARTIFACT_ZIP_PATH = OUTPUT_DIR / "government_parser_qwen3_4b_lora_artifact.zip"

for p in [FULL_TRAIN_OUTPUT_DIR, FINAL_ADAPTER_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def print_gpu_memory(label="GPU memory"):
    print("=" * 80)
    print(label)
    if not torch.cuda.is_available():
        print("CUDA not available")
        return
    for i in range(torch.cuda.device_count()):
        print(
            f"GPU {i}: {torch.cuda.get_device_name(i)} | "
            f"allocated={torch.cuda.memory_allocated(i)/1024**3:.3f}GB | "
            f"reserved={torch.cuda.memory_reserved(i)/1024**3:.3f}GB | "
            f"max={torch.cuda.max_memory_allocated(i)/1024**3:.3f}GB"
        )

print("BASE_MODEL_NAME:", BASE_MODEL_NAME)
print("TRAIN_MAX_LENGTH:", TRAIN_MAX_LENGTH)
print("CUDA:", torch.cuda.is_available(), "device_count:", torch.cuda.device_count())
print_gpu_memory("Initial GPU memory")

In [ ]:
def find_source_dataset_dir() -> Path:
    candidates = []
    for root in Path("/kaggle/input").iterdir():
        if root.is_dir():
            matches = list(root.rglob("parser_train.v1.jsonl"))
            if matches:
                candidates.append(matches[0].parent)
    if not candidates:
        raise FileNotFoundError("Không tìm thấy parser_train.v1.jsonl trong /kaggle/input.")
    return candidates[0]

SOURCE_DATASET_DIR = find_source_dataset_dir()

required_raw_files = [
    "parser_train.v1.jsonl",
    "parser_validation.v1.jsonl",
    "parser_test.v1.jsonl",
]

required_config_files = [
    "parsed_query_schema.v1.json",
    "parser_intents.v1.json",
    "parser_enums.v1.json",
    "question_families.v1.json",
    "country_catalog.v1.json",
    "indicator_catalog.v1.json",
    "analytics_metadata.v1.json",
    "parser_final_report.v1.json",
    "parser_split_report.v1.json",
    "parser_full_report.v1.json",
]

for filename in required_raw_files:
    shutil.copy2(SOURCE_DATASET_DIR / filename, RAW_DATA_DIR / filename)

for filename in required_config_files:
    src = SOURCE_DATASET_DIR / filename
    if src.exists():
        shutil.copy2(src, CONFIG_DIR / filename)

print("SOURCE_DATASET_DIR:", SOURCE_DATASET_DIR)
print("Copied dataset/config files.")

In [ ]:
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

SCHEMA = load_json(SCHEMA_PATH)
INTENTS = set(load_json(INTENTS_PATH))
ENUMS = load_json(ENUMS_PATH)
QUESTION_FAMILIES = load_json(QUESTION_FAMILIES_PATH)
COUNTRY_CATALOG = load_json(COUNTRY_CATALOG_PATH)
INDICATOR_CATALOG = load_json(INDICATOR_CATALOG_PATH)

FAMILY_TO_INTENT = {item["id"]: item["intent"] for item in QUESTION_FAMILIES["families"]}
COUNTRY_CODES = {item["code"] for item in COUNTRY_CATALOG["countries"]}
INDICATOR_CODES = {item["code"] for item in INDICATOR_CATALOG["indicators"]}
validator = jsonschema.Draft7Validator(SCHEMA)

def validate_parsed_query(parsed: dict) -> list[str]:
    errors = []

    for error in sorted(validator.iter_errors(parsed), key=lambda e: e.path):
        path = ".".join(str(p) for p in error.path)
        errors.append(f"schema_error:{path}:{error.message}")

    intent = parsed.get("intent")
    family = parsed.get("question_family")

    if intent not in INTENTS:
        errors.append(f"invalid_intent:{intent}")

    if family not in FAMILY_TO_INTENT:
        errors.append(f"invalid_question_family:{family}")
    elif intent != FAMILY_TO_INTENT[family]:
        errors.append(f"family_intent_mismatch:{family}:{intent}:{FAMILY_TO_INTENT[family]}")

    for code in parsed.get("indicators", []):
        if code not in INDICATOR_CODES:
            errors.append(f"invalid_indicator:{code}")

    for code in parsed.get("countries", []):
        if code not in COUNTRY_CODES:
            errors.append(f"invalid_country:{code}")

    for field in ["ranking_order", "chart_preference", "aggregation", "relative_time", "event_time"]:
        if parsed.get(field) not in set(ENUMS.get(field, [])):
            errors.append(f"invalid_enum:{field}:{parsed.get(field)}")

    for group in parsed.get("country_groups", []):
        if group not in set(ENUMS.get("country_groups", [])):
            errors.append(f"invalid_country_group:{group}")

    return errors

print("Config loaded:", len(INTENTS), "intents,", len(FAMILY_TO_INTENT), "families")

In [ ]:
def compact_json_string(obj: dict) -> str:
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"), sort_keys=False)

def convert_raw_to_sft(raw_path: Path, sft_path: Path, split_name: str):
    total = 0
    written = 0
    errors = []

    with raw_path.open("r", encoding="utf-8") as f_in, sft_path.open("w", encoding="utf-8") as f_out:
        for line_no, line in enumerate(f_in, start=1):
            total += 1
            try:
                row = json.loads(line)
                messages = row["messages"]

                system_content = messages[0].get("content") or SYSTEM_PROMPT
                user_content = messages[1]["content"]
                assistant_json = row.get("assistant_json") or json.loads(messages[2]["content"])

                parsed_from_message = json.loads(messages[2]["content"])
                if assistant_json != parsed_from_message:
                    raise ValueError("assistant_json mismatch")

                if validate_parsed_query(assistant_json):
                    raise ValueError("assistant_json schema/catalog validation failed")

                out = {
                    "prompt": [
                        {"role": "system", "content": system_content},
                        {"role": "user", "content": user_content},
                    ],
                    "completion": [
                        {"role": "assistant", "content": compact_json_string(assistant_json)}
                    ],
                    "sample_id": row.get("sample_id"),
                    "source_sample_id": row.get("source_sample_id"),
                    "plan_group_id": row.get("plan_group_id"),
                    "version": row.get("version"),
                    "split": split_name,
                    "generation_source": row.get("generation_source"),
                    "intent": row.get("intent"),
                    "question_family": row.get("question_family"),
                    "language_style": row.get("language_style"),
                }
                f_out.write(json.dumps(out, ensure_ascii=False) + "\n")
                written += 1

            except Exception as e:
                if len(errors) < 10:
                    errors.append({"line_no": line_no, "error": str(e)})

    if errors:
        raise RuntimeError(f"Convert failed for {split_name}: {errors}")

    return {"split": split_name, "total": total, "written": written}

conversion_reports = [
    convert_raw_to_sft(RAW_TRAIN_PATH, SFT_TRAIN_PATH, "train"),
    convert_raw_to_sft(RAW_VALIDATION_PATH, SFT_VALIDATION_PATH, "validation"),
    convert_raw_to_sft(RAW_TEST_PATH, SFT_TEST_PATH, "test"),
]

print(conversion_reports)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True, trust_remote_code=False)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

data_files = {
    "train": str(SFT_TRAIN_PATH),
    "validation": str(SFT_VALIDATION_PATH),
    "test": str(SFT_TEST_PATH),
}

raw_sft_dataset = load_dataset("json", data_files=data_files)

def add_training_text(example):
    messages = example["prompt"] + example["completion"]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

def add_token_length(example):
    ids = tokenizer(example["text"], add_special_tokens=False, truncation=False)["input_ids"]
    return {"text_token_length": len(ids)}

sft_dataset = raw_sft_dataset.map(add_training_text, desc="Render chat template")
sft_dataset = sft_dataset.map(add_token_length, desc="Count tokens")

too_long = {
    split: sum(1 for x in sft_dataset[split]["text_token_length"] if x > TRAIN_MAX_LENGTH)
    for split in ["train", "validation", "test"]
}

if any(v > 0 for v in too_long.values()):
    raise ValueError(f"Found samples longer than TRAIN_MAX_LENGTH={TRAIN_MAX_LENGTH}: {too_long}")

print("Dataset rows:", {split: len(sft_dataset[split]) for split in sft_dataset.keys()})
print("Too long:", too_long)

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

COMPUTE_DTYPE = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

model_kwargs = dict(
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=False,
    attn_implementation="sdpa",
)

try:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        dtype=COMPUTE_DTYPE,
        **model_kwargs,
    )
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=COMPUTE_DTYPE,
        **model_kwargs,
    )

model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

for _, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.zero_grad(set_to_none=True)

model.print_trainable_parameters()
print_gpu_memory("After model + LoRA load")

In [ ]:
from transformers import TrainerCallback

full_train_dataset = sft_dataset["train"]
train_eval_dataset = sft_dataset["validation"].select(range(min(512, len(sft_dataset["validation"]))))

# In Kaggle Commit, tqdm/progress bars often do not stream well.
# This callback prints plain stdout lines with flush=True, so Commit logs show progress.
COMMIT_TRAIN_LOG_PATH = REPORT_DIR / "commit_train_progress.log"

class CommitProgressCallback(TrainerCallback):
    def __init__(self, total_train_rows, per_device_batch_size, grad_accum_steps, epochs, log_path):
        self.total_train_rows = total_train_rows
        self.per_device_batch_size = per_device_batch_size
        self.grad_accum_steps = grad_accum_steps
        self.epochs = epochs
        self.expected_steps = math.ceil(total_train_rows / (per_device_batch_size * grad_accum_steps)) * epochs
        self.log_path = Path(log_path)
        self.train_start_time = None
        self.last_print_step = 0

    def _write(self, message: str):
        print(message, flush=True)
        with self.log_path.open("a", encoding="utf-8") as f:
            f.write(message + "\n")

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start_time = time.time()
        self.log_path.parent.mkdir(parents=True, exist_ok=True)
        self.log_path.write_text("", encoding="utf-8")
        max_steps = state.max_steps or self.expected_steps
        self._write(
            f"[TRAIN_BEGIN] rows={self.total_train_rows} expected_steps~={self.expected_steps} "
            f"state.max_steps={max_steps} batch={self.per_device_batch_size} "
            f"grad_accum={self.grad_accum_steps} epochs={self.epochs}"
        )

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return

        step = state.global_step
        max_steps = state.max_steps or self.expected_steps
        elapsed = time.time() - self.train_start_time if self.train_start_time else 0.0
        steps_per_sec = step / elapsed if elapsed > 0 and step > 0 else 0.0
        remaining_steps = max(max_steps - step, 0)
        eta_sec = remaining_steps / steps_per_sec if steps_per_sec > 0 else 0.0

        loss = logs.get("loss")
        lr = logs.get("learning_rate")
        epoch = logs.get("epoch")
        grad_norm = logs.get("grad_norm")

        parts = [
            f"[TRAIN_LOG] step={step}/{max_steps}",
            f"epoch={epoch:.4f}" if isinstance(epoch, (int, float)) else f"epoch={epoch}",
            f"elapsed={elapsed/60:.1f}m",
            f"eta={eta_sec/60:.1f}m",
            f"steps_per_sec={steps_per_sec:.4f}",
        ]

        if loss is not None:
            parts.append(f"loss={float(loss):.6f}")
        if lr is not None:
            parts.append(f"lr={float(lr):.2e}")
        if grad_norm is not None:
            try:
                parts.append(f"grad_norm={float(grad_norm):.4f}")
            except Exception:
                parts.append(f"grad_norm={grad_norm}")

        self._write(" | ".join(parts))

    def on_save(self, args, state, control, **kwargs):
        self._write(f"[SAVE] step={state.global_step} output_dir={args.output_dir}")

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.train_start_time if self.train_start_time else 0.0
        self._write(f"[TRAIN_END] step={state.global_step} elapsed={elapsed/60:.1f}m")

estimated_total_steps = math.ceil(
    len(full_train_dataset) / (PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
) * NUM_TRAIN_EPOCHS

print("=" * 100, flush=True)
print("Full train config", flush=True)
print("Train rows:", len(full_train_dataset), flush=True)
print("Estimated optimizer steps:", estimated_total_steps, flush=True)
print("Batch size:", PER_DEVICE_TRAIN_BATCH_SIZE, flush=True)
print("Grad accumulation:", GRADIENT_ACCUMULATION_STEPS, flush=True)
print("Logging every steps: 10", flush=True)
print("Commit train log file:", COMMIT_TRAIN_LOG_PATH, flush=True)
print("=" * 100, flush=True)

training_args = SFTConfig(
    output_dir=str(FULL_TRAIN_OUTPUT_DIR),

    dataset_text_field="text",
    max_length=TRAIN_MAX_LENGTH,
    packing=False,

    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=-1,

    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,

    fp16=False,
    bf16=False,
    max_grad_norm=0.0,

    logging_strategy="steps",
    logging_first_step=True,
    logging_steps=10,
    disable_tqdm=True,

    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=1,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    report_to="none",
    remove_unused_columns=True,
    seed=42,
)

progress_callback = CommitProgressCallback(
    total_train_rows=len(full_train_dataset),
    per_device_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    grad_accum_steps=GRADIENT_ACCUMULATION_STEPS,
    epochs=NUM_TRAIN_EPOCHS,
    log_path=COMMIT_TRAIN_LOG_PATH,
)

print("[START] Creating full SFTTrainer", flush=True)
trainer_create_start = time.time()
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=full_train_dataset,
    eval_dataset=None,
    processing_class=tokenizer,
    callbacks=[progress_callback],
)
print(f"[DONE] Creating full SFTTrainer | seconds={time.time() - trainer_create_start:.1f}", flush=True)

print_gpu_memory("Before full train")
print("[START] Full LoRA train", flush=True)
train_start_time = time.time()
train_result = trainer.train()
train_elapsed = round(time.time() - train_start_time, 2)
print("[DONE] Full LoRA train | seconds:", train_elapsed, flush=True)
print(train_result, flush=True)
print_gpu_memory("After full train")

trainer.model.save_pretrained(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))

train_report = {
    "base_model_name": BASE_MODEL_NAME,
    "train_max_length": TRAIN_MAX_LENGTH,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "estimated_total_steps": estimated_total_steps,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "final_adapter_dir": str(FINAL_ADAPTER_DIR),
    "commit_train_log_path": str(COMMIT_TRAIN_LOG_PATH),
    "train_result": str(train_result),
    "train_runtime_seconds_wall_clock": train_elapsed,
    "log_history": trainer.state.log_history,
}

with (REPORT_DIR / "phase9_full_train_report.json").open("w", encoding="utf-8") as f:
    json.dump(train_report, f, ensure_ascii=False, indent=2, default=str)

print("Saved final adapter:", FINAL_ADAPTER_DIR, flush=True)
print("Saved train progress log:", COMMIT_TRAIN_LOG_PATH, flush=True)


In [ ]:
SAFE_INTENTS = {"NEED_CLARIFICATION", "OFF_TOPIC", "UNSUPPORTED"}

def extract_first_json_object(text: str):
    text = (text or "").strip()
    try:
        return json.loads(text), text
    except Exception:
        pass

    start = text.find("{")
    if start < 0:
        raise ValueError("no_json_object_start")

    in_string = False
    escape = False
    depth = 0

    for i in range(start, len(text)):
        ch = text[i]

        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:i+1]
                return json.loads(candidate), candidate

    raise ValueError("no_complete_json_object")

def list_prf(golds, preds):
    tp = fp = fn = 0
    for gold, pred in zip(golds, preds):
        g = set(gold or [])
        p = set(pred or [])
        tp += len(g & p)
        fp += len(p - g)
        fn += len(g - p)
    precision = tp / (tp + fp) if tp + fp else 1.0
    recall = tp / (tp + fn) if tp + fn else 1.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

def safe_parse_prediction(text: str):
    try:
        parsed, extracted = extract_first_json_object(text)
        json_ok = True
        json_error = None
    except Exception as e:
        return {
            "parsed": None,
            "extracted": None,
            "valid_json": False,
            "schema_pass": False,
            "json_error": str(e),
            "schema_errors": ["invalid_json"],
        }

    schema_errors = validate_parsed_query(parsed)
    return {
        "parsed": parsed,
        "extracted": extracted,
        "valid_json": json_ok,
        "schema_pass": len(schema_errors) == 0,
        "json_error": json_error,
        "schema_errors": schema_errors,
    }

def is_safe_parse(parsed):
    if not isinstance(parsed, dict):
        return True
    return (
        parsed.get("needs_clarification") is True
        or parsed.get("intent") in SAFE_INTENTS
    )

def is_dangerous_wrong_query(gold, pred, schema_pass):
    if not schema_pass or not isinstance(pred, dict):
        return False

    pred_safe = is_safe_parse(pred)
    gold_safe = is_safe_parse(gold)

    if pred_safe:
        return False

    if gold_safe and not pred_safe:
        return True

    key_fields = [
        "intent",
        "question_family",
        "indicators",
        "countries",
        "country_groups",
        "start_year",
        "end_year",
        "relative_time",
        "event_time",
        "ranking_order",
        "limit",
        "threshold",
        "aggregation",
    ]

    for field in key_fields:
        if pred.get(field) != gold.get(field):
            return True

    return False

def compute_metrics(rows):
    total = len(rows)
    valid_json = sum(1 for r in rows if r["valid_json"])
    schema_pass = sum(1 for r in rows if r["schema_pass"])

    schema_rows = [r for r in rows if r["schema_pass"]]
    exact_match = sum(
        1 for r in rows
        if r["schema_pass"] and r["pred"] == r["gold"]
    )

    def acc_field(field):
        denom = len(schema_rows)
        if denom == 0:
            return 0.0
        return sum(1 for r in schema_rows if r["pred"].get(field) == r["gold"].get(field)) / denom

    indicator_prf = list_prf(
        [r["gold"].get("indicators", []) for r in schema_rows],
        [r["pred"].get("indicators", []) for r in schema_rows],
    )
    country_prf = list_prf(
        [r["gold"].get("countries", []) for r in schema_rows],
        [r["pred"].get("countries", []) for r in schema_rows],
    )
    country_group_prf = list_prf(
        [r["gold"].get("country_groups", []) for r in schema_rows],
        [r["pred"].get("country_groups", []) for r in schema_rows],
    )

    dangerous = sum(
        1 for r in rows
        if is_dangerous_wrong_query(r["gold"], r["pred"], r["schema_pass"])
    )

    executable = sum(
        1 for r in rows
        if r["schema_pass"] and not is_safe_parse(r["pred"])
    )

    metrics = {
        "total": total,
        "valid_json_rate": valid_json / total if total else 0,
        "schema_pass_rate": schema_pass / total if total else 0,
        "exact_full_json_rate": exact_match / total if total else 0,
        "intent_accuracy_on_schema_pass": acc_field("intent"),
        "question_family_accuracy_on_schema_pass": acc_field("question_family"),
        "year_exact_accuracy_on_schema_pass": (
            sum(
                1 for r in schema_rows
                if r["pred"].get("start_year") == r["gold"].get("start_year")
                and r["pred"].get("end_year") == r["gold"].get("end_year")
            ) / len(schema_rows)
            if schema_rows else 0.0
        ),
        "ranking_order_accuracy_on_schema_pass": acc_field("ranking_order"),
        "aggregation_accuracy_on_schema_pass": acc_field("aggregation"),
        "chart_preference_accuracy_on_schema_pass": acc_field("chart_preference"),
        "needs_clarification_accuracy_on_schema_pass": acc_field("needs_clarification"),
        "indicator_micro": indicator_prf,
        "country_micro": country_prf,
        "country_group_micro": country_group_prf,
        "executable_plan_rate": executable / total if total else 0,
        "dangerous_wrong_query_rate": dangerous / total if total else 0,
        "dangerous_wrong_query_count": dangerous,
        "schema_error_top": dict(Counter(err for r in rows for err in r["schema_errors"]).most_common(30)),
    }
    return metrics

In [ ]:
model_for_eval = trainer.model
model_for_eval.eval()
try:
    model_for_eval.gradient_checkpointing_disable()
except Exception:
    pass
model_for_eval.config.use_cache = True
tokenizer.padding_side = "left"
print("Eval max samples:", EVAL_MAX_SAMPLES, "| batch size:", EVAL_BATCH_SIZE, "| padding_side:", tokenizer.padding_side, flush=True)

def iter_batches(dataset, batch_size):
    for start in range(0, len(dataset), batch_size):
        end = min(start + batch_size, len(dataset))
        yield start, dataset.select(range(start, end))

def generate_for_batch(batch):
    prompts = [
        tokenizer.apply_chat_template(
            row["prompt"],
            tokenize=False,
            add_generation_prompt=True,
        )
        for row in batch
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=TRAIN_MAX_LENGTH,
        add_special_tokens=False,
    )

    first_device = next(model_for_eval.parameters()).device
    inputs = {k: v.to(first_device) for k, v in inputs.items()}

    with torch.no_grad():
        generated = model_for_eval.generate(
            **inputs,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    results = []
    input_len = inputs["input_ids"].shape[1]

    for i in range(generated.shape[0]):
        new_tokens = generated[i][input_len:]
        decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        results.append(decoded)

    return results

def evaluate_split(split_name: str, dataset, max_samples=None):
    if max_samples is not None:
        dataset = dataset.select(range(min(max_samples, len(dataset))))

    rows = []
    prediction_path = REPORT_DIR / f"{split_name}_predictions.jsonl"

    start_time = time.time()

    with prediction_path.open("w", encoding="utf-8") as f_out:
        for start, batch_ds in iter_batches(dataset, EVAL_BATCH_SIZE):
            batch = [batch_ds[i] for i in range(len(batch_ds))]
            outputs = generate_for_batch(batch)

            for row, output in zip(batch, outputs):
                gold = json.loads(row["completion"][0]["content"])
                parsed_info = safe_parse_prediction(output)
                pred = parsed_info["parsed"]

                result = {
                    "sample_id": row.get("sample_id"),
                    "generation_source": row.get("generation_source"),
                    "intent": row.get("intent"),
                    "question_family": row.get("question_family"),
                    "user_message": row["prompt"][1]["content"],
                    "gold": gold,
                    "raw_output": output,
                    "pred": pred,
                    "valid_json": parsed_info["valid_json"],
                    "schema_pass": parsed_info["schema_pass"],
                    "json_error": parsed_info["json_error"],
                    "schema_errors": parsed_info["schema_errors"],
                }

                rows.append(result)
                f_out.write(json.dumps(result, ensure_ascii=False) + "\n")

            if (start // EVAL_BATCH_SIZE) % 25 == 0:
                done = min(start + EVAL_BATCH_SIZE, len(dataset))
                elapsed = max(time.time() - start_time, 1e-6)
                speed = done / elapsed
                remaining = (len(dataset) - done) / speed if speed > 0 else 0
                print(f"{split_name}: {done}/{len(dataset)} | {speed:.2f} samples/sec | ETA {remaining/60:.1f} min", flush=True)

    metrics = compute_metrics(rows)
    metrics["split"] = split_name
    metrics["num_samples"] = len(dataset)
    metrics["runtime_seconds"] = round(time.time() - start_time, 2)
    metrics["prediction_path"] = str(prediction_path)

    metrics_path = REPORT_DIR / f"{split_name}_metrics.json"
    with metrics_path.open("w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    return metrics

validation_metrics = evaluate_split("validation", sft_dataset["validation"], EVAL_MAX_SAMPLES)
test_metrics = evaluate_split("test", sft_dataset["test"], EVAL_MAX_SAMPLES)

print("Validation metrics:")
print(json.dumps(validation_metrics, ensure_ascii=False, indent=2))

print("Test metrics:")
print(json.dumps(test_metrics, ensure_ascii=False, indent=2))

In [ ]:
SMOKE_TESTS = [
    "So sánh nợ công Việt Nam và Thái Lan từ 2010 đến 2023.",
    "Top 10 nước có lạm phát CPI cao nhất năm 2020.",
    "Xu hướng thất nghiệp của Nhật Bản sau COVID.",
    "Việt Nam có dữ liệu nợ công từ năm nào đến năm nào?",
    "Vẽ line chart cho tăng trưởng GDP thực của Indonesia giai đoạn 2000-2022.",
    "Nước nào trong ASEAN có tỷ lệ nghèo giảm mạnh nhất?",
    "Dự báo nợ công Việt Nam bằng ARIMA.",
    "Viết SQL lấy bảng gold_fiscal_monetary.",
    "Việt Nam thế nào?",
    "So sánh Việt Nam với Thái Lan.",
]

def parse_one(message: str):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": message},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    first_device = next(model_for_eval.parameters()).device
    inputs = {k: v.to(first_device) for k, v in inputs.items()}

    with torch.no_grad():
        generated = model_for_eval.generate(
            **inputs,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = generated[0][inputs["input_ids"].shape[-1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    parsed_info = safe_parse_prediction(decoded)
    return {
        "message": message,
        "raw_output": decoded,
        "pred": parsed_info["parsed"],
        "valid_json": parsed_info["valid_json"],
        "schema_pass": parsed_info["schema_pass"],
        "schema_errors": parsed_info["schema_errors"],
    }

smoke_results = [parse_one(q) for q in SMOKE_TESTS]

SMOKE_TEST_PATH = REPORT_DIR / "manual_smoke_test_results.json"
with SMOKE_TEST_PATH.open("w", encoding="utf-8") as f:
    json.dump(smoke_results, f, ensure_ascii=False, indent=2)

for item in smoke_results:
    print("=" * 100)
    print("USER:", item["message"])
    print("SCHEMA_PASS:", item["schema_pass"])
    print("OUTPUT:", item["raw_output"])
    if item["schema_errors"]:
        print("ERRORS:", item["schema_errors"][:5])

In [ ]:
manifest = {
    "artifact_name": "government_parser_qwen3_4b_lora",
    "base_model_name": BASE_MODEL_NAME,
    "adapter_dir": str(FINAL_ADAPTER_DIR),
    "train_max_length": TRAIN_MAX_LENGTH,
    "eval_config": {
        "eval_max_samples": EVAL_MAX_SAMPLES,
        "eval_batch_size": EVAL_BATCH_SIZE,
    },
    "generation": {
        "do_sample": False,
        "max_new_tokens": GEN_MAX_NEW_TOKENS,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    },
    "train_config": {
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        "quantization": "4bit_nf4_double_quant",
    },
    "metrics": {
        "validation": validation_metrics,
        "test": test_metrics,
    },
    "files": {
        "validation_metrics": str(REPORT_DIR / "validation_metrics.json"),
        "test_metrics": str(REPORT_DIR / "test_metrics.json"),
        "smoke_test": str(SMOKE_TEST_PATH),
    },
}

MANIFEST_PATH = EXPORT_DIR / "model_manifest.json"
with MANIFEST_PATH.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=str)

def add_dir_to_zip(zipf, src_dir: Path, arc_prefix: str):
    for path in src_dir.rglob("*"):
        if path.is_file():
            zipf.write(path, arcname=str(Path(arc_prefix) / path.relative_to(src_dir)))

if ARTIFACT_ZIP_PATH.exists():
    ARTIFACT_ZIP_PATH.unlink()

with zipfile.ZipFile(ARTIFACT_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    add_dir_to_zip(zipf, FINAL_ADAPTER_DIR, "final_adapter")
    add_dir_to_zip(zipf, CONFIG_DIR, "configs")
    add_dir_to_zip(zipf, REPORT_DIR, "reports")
    zipf.write(MANIFEST_PATH, arcname="model_manifest.json")

print("Final adapter:", FINAL_ADAPTER_DIR)
print("Manifest:", MANIFEST_PATH)
print("Artifact zip:", ARTIFACT_ZIP_PATH)
print("Artifact size MB:", round(ARTIFACT_ZIP_PATH.stat().st_size / 1024**2, 2))

In [ ]:
CLEANUP_HF_CACHE = True
CLEANUP_TRAIN_CHECKPOINTS = False
if CLEANUP_HF_CACHE and HF_CACHE_DIR.exists():
    shutil.rmtree(HF_CACHE_DIR, ignore_errors=True)
    print("Removed HF cache:", HF_CACHE_DIR)

if CLEANUP_TRAIN_CHECKPOINTS and FULL_TRAIN_OUTPUT_DIR.exists():
    shutil.rmtree(FULL_TRAIN_OUTPUT_DIR, ignore_errors=True)
    print("Removed train checkpoints:", FULL_TRAIN_OUTPUT_DIR)

total, used, free = shutil.disk_usage("/kaggle/working")
print("Disk free GB:", round(free / 1024**3, 2))
print("Done. Commit/save this Kaggle notebook output.")